# Out-of-Context Cross-Encoder Experiment

Avaliacao zero-shot de um MiniLM multilingue que pontua diretamente a relevancia do par `system_prompt` e `user_prompt`, sem fine-tuning.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import CrossEncoder
from sklearn.calibration import CalibrationDisplay
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
MAX_LENGTH = 256
PREDICTION_BATCH_SIZE = 32
LATENCY_REPEATS = 5

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in (cwd, *cwd.parents) if (path / "data").exists())
RESULTS_DIR = PROJECT_ROOT / "experiments" / "out_of_context_cross_encoder" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = PROJECT_ROOT / "data" / "out-of-context.parquet"


def prompt_pairs(frame):
    return list(zip(frame["system_prompt"], frame["user_prompt"], strict=True))


def out_of_context_scores(model, frame, show_progress_bar=False):
    relevance_scores = model.predict(
        prompt_pairs(frame),
        batch_size=PREDICTION_BATCH_SIZE,
        show_progress_bar=show_progress_bar,
        activation_fn=torch.nn.Identity(),
        convert_to_numpy=True,
    )
    return -np.asarray(relevance_scores, dtype=np.float64).reshape(-1)


def calibrated_probabilities(method, calibrator, scores):
    if method == "sigmoid":
        return calibrator.predict_proba(scores.reshape(-1, 1))[:, 1]
    return calibrator.predict(scores)


def metrics_at_threshold(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "false_positive_rate": fp / (fp + tn) if fp + tn else 0.0,
        "false_negative_rate": fn / (fn + tp) if fn + tp else 0.0,
        "false_positives": int(fp),
        "false_negatives": int(fn),
    }


def select_threshold(y_true, probabilities):
    table = pd.DataFrame(
        metrics_at_threshold(y_true, probabilities, threshold)
        for threshold in np.linspace(0.001, 0.999, 999)
    )
    operating_points = table[table["precision"].gt(0) & table["recall"].gt(0)]
    selected = (
        operating_points.assign(
            precision_recall_gap=(operating_points["precision"] - operating_points["recall"]).abs()
        )
        .sort_values(
            ["precision_recall_gap", "f1", "threshold"],
            ascending=[True, False, False],
        )
        .iloc[0]
    )
    return float(selected["threshold"]), table


def batched(values, batch_size):
    for start in range(0, len(values), batch_size):
        yield values[start:start + batch_size]


def tokenize_pairs(model, pairs):
    encoded_batches = []
    for batch in batched(pairs, PREDICTION_BATCH_SIZE):
        system_prompts, user_prompts = zip(*batch, strict=True)
        encoded = model.tokenizer(
            list(system_prompts),
            list(user_prompts),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded_batches.append({key: value.to(model.device) for key, value in encoded.items()})
    return encoded_batches


def measure_latency(model, frame, repeats=LATENCY_REPEATS):
    pairs = prompt_pairs(frame)
    warmup_pairs = pairs[:min(PREDICTION_BATCH_SIZE, len(pairs))]
    model.predict(warmup_pairs, batch_size=PREDICTION_BATCH_SIZE, show_progress_bar=False)

    preprocessing_durations = []
    for _ in range(repeats):
        started_at = perf_counter()
        tokenize_pairs(model, pairs)
        preprocessing_durations.append(perf_counter() - started_at)
    encoded_batches = tokenize_pairs(model, pairs)

    with torch.inference_mode():
        model.model(**encoded_batches[0])
    model_durations = []
    for _ in range(repeats):
        started_at = perf_counter()
        with torch.inference_mode():
            for encoded in encoded_batches:
                model.model(**encoded)
        model_durations.append(perf_counter() - started_at)

    end_to_end_durations = []
    for _ in range(repeats):
        started_at = perf_counter()
        model.predict(pairs, batch_size=PREDICTION_BATCH_SIZE, show_progress_bar=False)
        end_to_end_durations.append(perf_counter() - started_at)

    sample_count = len(frame)
    return {
        "preprocessing_latency_ms_per_sample": np.median(preprocessing_durations) * 1000 / sample_count,
        "model_latency_ms_per_sample": np.median(model_durations) * 1000 / sample_count,
        "end_to_end_latency_ms_per_sample": np.median(end_to_end_durations) * 1000 / sample_count,
    }

RESULTS_DIR

## Dataset

O mesmo dataset e as mesmas garantias de isolamento do experimento XGBoost sao usados para permitir comparacao direta.

In [ ]:
data = pd.read_parquet(DATA_PATH)
expected_columns = {"system_prompt", "user_prompt", "out_of_context", "split", "pair_id"}
assert expected_columns.issubset(data.columns)
assert set(data["out_of_context"].unique()) == {0, 1}

split_systems = {split: set(frame["system_prompt"]) for split, frame in data.groupby("split")}
assert split_systems["train"].isdisjoint(split_systems["validation"])
assert split_systems["train"].isdisjoint(split_systems["test"])
assert split_systems["validation"].isdisjoint(split_systems["test"])
split_pair_ids = {split: set(frame["pair_id"]) for split, frame in data.groupby("split")}
assert split_pair_ids["train"].isdisjoint(split_pair_ids["validation"])
assert split_pair_ids["train"].isdisjoint(split_pair_ids["test"])
assert split_pair_ids["validation"].isdisjoint(split_pair_ids["test"])
pairs = data.groupby("pair_id")["out_of_context"].agg(["count", "sum"])
assert (pairs["count"].eq(2) & pairs["sum"].eq(1)).all()

dataset_summary = data.groupby(["split", "out_of_context"]).size().rename("rows").reset_index()
print(f"Rows: {len(data):,}")
display(dataset_summary)

## Zero-shot model

O MiniLM multilingue pre-treinado para ranking recebe o par completo sem fine-tuning. Como scores maiores representam maior relevancia, o experimento inverte o score bruto para que valores maiores representem maior evidencia de out-of-context.

In [ ]:
validation_data = data[data["split"].eq("validation")].reset_index(drop=True)
test_data = data[data["split"].eq("test")].reset_index(drop=True)

validation_splitter = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
calibration_idx, threshold_idx = next(
    validation_splitter.split(
        validation_data,
        y=validation_data["out_of_context"],
        groups=validation_data["pair_id"],
    )
)
calibration_data = validation_data.iloc[calibration_idx].reset_index(drop=True)
threshold_data = validation_data.iloc[threshold_idx].reset_index(drop=True)

cross_encoder = CrossEncoder(
    MODEL_NAME,
    max_length=MAX_LENGTH,
)

model_configuration = pd.DataFrame([{
    "model": MODEL_NAME,
    "mode": "zero-shot",
    "score_orientation": "negative relevance score",
    "max_length": MAX_LENGTH,
    "prediction_batch_size": PREDICTION_BATCH_SIZE,
    "device": str(cross_encoder.device),
}])
display(model_configuration)

## Calibration and threshold selection

Metade da validacao transforma os scores zero-shot invertidos em probabilidades com sigmoid ou isotonic. A outra metade seleciona o threshold no ponto em que precision e recall sao iguais ou o mais proximos possivel.

In [ ]:
y_calibration = calibration_data["out_of_context"].to_numpy()
y_threshold = threshold_data["out_of_context"].to_numpy()
calibration_scores = out_of_context_scores(cross_encoder, calibration_data, show_progress_bar=True)
threshold_scores = out_of_context_scores(cross_encoder, threshold_data, show_progress_bar=True)

calibrators = {
    "sigmoid": LogisticRegression(random_state=SEED).fit(
        calibration_scores.reshape(-1, 1), y_calibration
    ),
    "isotonic": IsotonicRegression(out_of_bounds="clip").fit(
        calibration_scores, y_calibration
    ),
}
calibration_rows = []
for method, calibrator in calibrators.items():
    probabilities = calibrated_probabilities(method, calibrator, calibration_scores)
    calibration_rows.append({
        "model": MODEL_NAME,
        "mode": "zero-shot",
        "method": method,
        "brier_score": brier_score_loss(y_calibration, probabilities),
    })
calibration_comparison = pd.DataFrame(calibration_rows).sort_values("brier_score")
selected_calibration_method = calibration_comparison.iloc[0]["method"]
selected_calibrator = calibrators[selected_calibration_method]
threshold_probabilities = calibrated_probabilities(
    selected_calibration_method, selected_calibrator, threshold_scores
)
selected_threshold, threshold_table = select_threshold(y_threshold, threshold_probabilities)
threshold_metrics = metrics_at_threshold(y_threshold, threshold_probabilities, selected_threshold)
validation_comparison = pd.DataFrame([{
    "model": MODEL_NAME,
    "mode": "zero-shot",
    "calibration_method": selected_calibration_method,
    "brier_score": calibration_comparison.iloc[0]["brier_score"],
    "validation_pr_auc": average_precision_score(y_threshold, threshold_probabilities),
    "validation_roc_auc": roc_auc_score(y_threshold, threshold_probabilities),
    **threshold_metrics,
}])

fig, axis = plt.subplots(figsize=(8, 6))
CalibrationDisplay.from_predictions(
    y_threshold,
    threshold_probabilities,
    n_bins=10,
    strategy="quantile",
    name=f"Zero-shot cross-encoder / {selected_calibration_method}",
    ax=axis,
)
axis.set_title("Out-of-context zero-shot cross-encoder calibration")
axis.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "calibration_plot.png", dpi=160)
plt.show()

print(f"Selected calibration: {selected_calibration_method}")
print(f"Selected threshold: {selected_threshold:.3f}")
display(calibration_comparison)
display(validation_comparison)

## Test evaluation, latency and exports

Avaliacao final no teste com as mesmas metricas do experimento baseado em features. A latencia separa tokenizacao, forward do Transformer e execucao end-to-end.

In [ ]:
y_test = test_data["out_of_context"].to_numpy()
test_scores = out_of_context_scores(cross_encoder, test_data, show_progress_bar=True)
test_probabilities = calibrated_probabilities(
    selected_calibration_method, selected_calibrator, test_scores
)
latency_metrics = measure_latency(cross_encoder, test_data)
test_metrics = metrics_at_threshold(y_test, test_probabilities, selected_threshold)
model_comparison = pd.DataFrame([{
    "model": MODEL_NAME,
    "mode": "zero-shot",
    "calibration_method": selected_calibration_method,
    **test_metrics,
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "pr_auc": average_precision_score(y_test, test_probabilities),
    **latency_metrics,
}])

predictions = (test_probabilities >= selected_threshold).astype(int)
count_matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
percentage_matrix = confusion_matrix(y_test, predictions, labels=[0, 1], normalize="true")
labels = ["In context", "Out of context"]

fig, axis = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(count_matrix, display_labels=labels).plot(
    ax=axis, cmap="Blues", colorbar=False, values_format="d"
)
axis.set_title("Zero-shot cross-encoder - confusion matrix (counts)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_matrix_counts.png", dpi=160)
plt.show()

fig, axis = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(percentage_matrix, display_labels=labels).plot(
    ax=axis, cmap="Blues", colorbar=False, values_format=".1%"
)
axis.set_title("Zero-shot cross-encoder - confusion matrix (row %)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_matrix_percentages.png", dpi=160)
plt.show()

fig, axis = plt.subplots(figsize=(8, 5))
axis.plot(threshold_table["threshold"], threshold_table["recall"], label="Recall")
axis.plot(threshold_table["threshold"], threshold_table["precision"], label="Precision")
axis.plot(threshold_table["threshold"], threshold_table["false_positive_rate"], label="FPR")
axis.axvline(
    selected_threshold, color="black", linestyle="--",
    label=f"Selected: {selected_threshold:.3f}",
)
axis.set(
    xlabel="Threshold", ylabel="Metric", ylim=(0, 1.02),
    title="Zero-shot cross-encoder threshold analysis (validation)",
)
axis.grid(alpha=0.25)
axis.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "threshold_analysis.png", dpi=160)
plt.show()

confusion_counts = pd.DataFrame(
    count_matrix,
    index=["actual_in_context", "actual_out_of_context"],
    columns=["predicted_in_context", "predicted_out_of_context"],
)
confusion_percentages = pd.DataFrame(
    percentage_matrix,
    index=["actual_in_context", "actual_out_of_context"],
    columns=["predicted_in_context", "predicted_out_of_context"],
)
with pd.ExcelWriter(RESULTS_DIR / "results.xlsx", engine="openpyxl") as writer:
    dataset_summary.to_excel(writer, sheet_name="dataset_distribution", index=False)
    model_configuration.to_excel(writer, sheet_name="model_configuration", index=False)
    calibration_comparison.to_excel(writer, sheet_name="calibration", index=False)
    validation_comparison.to_excel(writer, sheet_name="validation_comparison", index=False)
    model_comparison.to_excel(writer, sheet_name="test_comparison", index=False)
    threshold_table.to_excel(writer, sheet_name="threshold_analysis", index=False)
    confusion_counts.to_excel(writer, sheet_name="confusion_counts")
    confusion_percentages.to_excel(writer, sheet_name="confusion_percentages")

display(model_comparison)
display(confusion_counts)
display(confusion_percentages.style.format("{:.1%}"))

In [ ]:
expected_results = {
    RESULTS_DIR / "calibration_plot.png",
    RESULTS_DIR / "confusion_matrix_counts.png",
    RESULTS_DIR / "confusion_matrix_percentages.png",
    RESULTS_DIR / "threshold_analysis.png",
    RESULTS_DIR / "results.xlsx",
}
missing_results = [path for path in expected_results if not path.exists()]
assert not missing_results, f"Missing results: {missing_results}"
assert model_comparison[[
    "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"
]].apply(lambda column: column.between(0.0, 1.0).all()).all()
assert model_comparison[[
    "preprocessing_latency_ms_per_sample",
    "model_latency_ms_per_sample",
    "end_to_end_latency_ms_per_sample",
]].gt(0.0).all().all()
print("Experiment completed successfully")
print("\n".join(str(path.relative_to(PROJECT_ROOT)) for path in sorted(expected_results)))